#**UNIVERSIDAD ICESI**
# **MAESTRÍA EN INTELIGENCIA ARTIFICIAL APLICADA**  
## **Trabajo 5: Generación de Texto con modelos GPT**

### Integrantes:
- Diego Agudelo  
- Angelica Maria Mayor  
- Freddy Mauricio Gutierrez  
- Carlos Alberto Martinez Ramirez.  
- Wilman Quiñonez  

---
# Generación de Texto con modelos GPT

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cam2149/icesi-nlp/blob/Entrega5/Sesion5/Entrega5_18925.ipynb)

Este notebook realiza un experimento de generación de texto basado en modelos tipo GPT-2 preentrenados en español. Se utiliza un modelo GPT-2 para generar la letra de una canción a partir de un texto inicial. Posteriormente, se realiza un fine-tuning del modelo con un dataset especializado de letras de canciones en español para observar cómo cambia la generación. Finalmente, se genera un audio a partir de la letra obtenida para tener una representación más tangible del resultado.


#### Referencias
- Dataset: https://huggingface.co/datasets/josuealdana/latin_lyrics
- [Improving Language Understanding by Generative Pre-Training](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf)
- [Natural Language Processing with Transformers: Building Language Applications With Hugging Face](https://www.amazon.com/Natural-Language-Processing-Transformers-Applications/dp/1098103246)
- [GPT2 Spanish](https://huggingface.co/DeepESP/gpt2-spanish)
- [Fine-Tune a non-Englush GPT-2 Model with Huggingface](https://www.philschmid.de/fine-tune-a-non-english-gpt-2-model-with-huggingface)

In [27]:
# Importa la librería pkg_resources, que permite acceder a información
# sobre los paquetes instalados en el entorno de Python.
import pkg_resources
# Importa la librería warnings, la cual maneja advertencias (warnings)
# que pueden aparecer durante la ejecución del programa.
import warnings
# Configura el filtro de advertencias para que se ignoren todas.
# Esto significa que no se mostrarán mensajes de advertencia en la ejecución.
warnings.filterwarnings('ignore')
# Crea una lista con los nombres (KEY) de todos los paquetes instalados
# en el entorno actual, usando working_set que devuelve los paquetes activos.
installed_packages = [package.key for package in pkg_resources.working_set]
# Verifica si entre los paquetes instalados está 'google-colab'.
# Si existe, significa que el código se está ejecutando en Google Colab.
IN_COLAB = 'google-colab' in installed_packages


Instalación de versiones de python y librerias

In [28]:
#!test '{IN_COLAB}' = 'True' && wget  https://github.com/Ohtar10/icesi-nlp/raw/refs/heads/main/requirements.txt && pip install -r requirements.txt
!test '{IN_COLAB}' = 'True' && sudo apt-get update -y
!test '{IN_COLAB}' = 'True' && sudo apt-get install python3.10 python3.10-distutils python3.10-lib2to3 -y
!test '{IN_COLAB}' = 'True' && sudo update-alternatives --install /usr/local/bin/python python /usr/bin/python3.11 2
!test '{IN_COLAB}' = 'True' && sudo update-alternatives --install /usr/local/bin/python python /usr/bin/python3.10 1
!test '{IN_COLAB}' = 'True' && pip install lightning datasets 'transformers[torch]' sentence-transformers torchinfo evaluate

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

## Generative pre-training Transformer - GPT

![](https://github.com/Ohtar10/icesi-nlp/blob/main/assets/gpt.png?raw=1)

Los modelos tipo GPT, introducidos por Radfor, et.al., de OpenAI, al igual que los modelos BERT, hacen uso extensivo de la arquitectura de transformers como hemos estado viendo. Las diferencias claves se podrían resumir en:

1. GPT utiliza bloques de **Transformer Decoder** encadenados, mientras que el modelo BERT utiliza bloques de *Transformer Encoder*
2. GPT se centra en la generación de texto basado en un contexto, la tarea principal es la predicción del siguiente token en la secuencia, mientras que BERT se centra en el completado de partes de una secuencia, en función de un contexto anterior y posterior a la secuencia de entrada. Entonces BERT se centra en la construicción de representación de lenguage, mientras que GPT se centra en la generación de texto en función de un contexto.

Sin embargo, ambos se basan en la misma premisa de pre-entrenar el modelo en tareas no-supervisadas o semi-supervisadas para que el modelo aprenda las representaciones semánticas del lenguage y luego al modelo se le pueda hacer fine tuning a tareas posteriores.

Importa la librería torch, que es el framework principal para trabajar con tensores y redes neuronales en PyTorch.

In [29]:
import torch
# Importa la librería transformers, que contiene modelos de lenguaje preentrenados
# como BERT, GPT-2, etc., desarrollados por HuggingFace.
import transformers
# Importa de manera específica dos clases/funciones que se usarán en este script:
# - AutoModelForCausalLM: Para cargar un modelo de lenguaje causal (como GPT-2).
# - AutoTokenizer: Para cargar el tokenizador asociado al modelo.
from transformers import AutoModelForCausalLM, AutoTokenizer
# Define el dispositivo de ejecución:
# - "cuda" si hay una GPU disponible.
# - "cpu" en caso contrario.
device = "cuda" if torch.cuda.is_available() else "cpu"
# Nombre del modelo preentrenado a utilizar.
# En este caso se usa GPT-2 adaptado al español.
model_name = "DeepESP/gpt2-spanish"
# Carga el tokenizador asociado al modelo de HuggingFace.
# El tokenizador convierte texto en números (tokens) que el modelo entiende.
tokenizer = AutoTokenizer.from_pretrained(model_name)
# Carga el modelo preentrenado para tareas de generación de texto causal.
# Luego, se envía el modelo al dispositivo seleccionado (GPU o CPU).
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
# Muestra una representación del modelo cargado.
# Esto sirve para verificar que se cargó correctamente.
model


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

La función model.named_modules() recorre todos los módulos definidos dentro del modelo, incluyendo el propio modelo y cada uno de sus submódulos. Esta función devuelve un iterador de tuplas con la forma (nombre, módulo).

In [30]:
# Aquí, solo se extrae el nombre de cada módulo y se guarda en la lista 'modules'
modules = [m for m, _ in model.named_modules()]
# Muestra la lista de nombres de todos los módulos (capas y subcapas) presentes en el modelo.
modules

['',
 'transformer',
 'transformer.wte',
 'transformer.wpe',
 'transformer.drop',
 'transformer.h',
 'transformer.h.0',
 'transformer.h.0.ln_1',
 'transformer.h.0.attn',
 'transformer.h.0.attn.c_attn',
 'transformer.h.0.attn.c_proj',
 'transformer.h.0.attn.attn_dropout',
 'transformer.h.0.attn.resid_dropout',
 'transformer.h.0.ln_2',
 'transformer.h.0.mlp',
 'transformer.h.0.mlp.c_fc',
 'transformer.h.0.mlp.c_proj',
 'transformer.h.0.mlp.act',
 'transformer.h.0.mlp.dropout',
 'transformer.h.1',
 'transformer.h.1.ln_1',
 'transformer.h.1.attn',
 'transformer.h.1.attn.c_attn',
 'transformer.h.1.attn.c_proj',
 'transformer.h.1.attn.attn_dropout',
 'transformer.h.1.attn.resid_dropout',
 'transformer.h.1.ln_2',
 'transformer.h.1.mlp',
 'transformer.h.1.mlp.c_fc',
 'transformer.h.1.mlp.c_proj',
 'transformer.h.1.mlp.act',
 'transformer.h.1.mlp.dropout',
 'transformer.h.2',
 'transformer.h.2.ln_1',
 'transformer.h.2.attn',
 'transformer.h.2.attn.c_attn',
 'transformer.h.2.attn.c_proj',
 'tran

Observermos un ejemplo de generación simple.

In [31]:
text = "Había una vez"
best = 10

with torch.no_grad():
    tokens = tokenizer(text, return_tensors='pt')['input_ids'].to(device)
    print("Dimensiones de la entrada:", tokens.shape)
    output = model(input_ids=tokens)
    print("Dimensiones de la salida:", output.logits.shape)
    output = output.logits[0, -1, :]
    print("Dimensiones del último token de la secuencia:", output.shape)
    probs = torch.softmax(output, dim=-1)
    print("Dimensiones de la probabilidad de los tokens:", probs.shape)
    sorted_probs = torch.argsort(probs, dim=-1, descending=True)
    print({tokenizer.decode(token): f"{prob.cpu().numpy() * 100:.2f}%" for token, prob in zip(sorted_probs[:best], probs[sorted_probs[:best]])})

Dimensiones de la entrada: torch.Size([1, 3])
Dimensiones de la salida: torch.Size([1, 3, 50257])
Dimensiones del último token de la secuencia: torch.Size([50257])
Dimensiones de la probabilidad de los tokens: torch.Size([50257])
{' más': '40.87%', ' que': '10.55%', ' en': '7.27%', ',': '4.90%', ' allí': '0.99%', ' se': '0.87%', '.': '0.82%', ' dentro': '0.81%', ' a': '0.78%', ' al': '0.78%'}


## Implementando una función de generación

Ahora, la idea es que este modelo nos sirva para generar texto de forma recurrente e incremental. En la última capa de los modelos tipo GPT encontrarémos un tensor con forma $(b, s, v)$, donde:

- $b$: Es el tamaño del bache, o la cantidad de secuencias a procesar.
- $s$: Es la longitud de la secuencia de entrada.
- $v$: Es el tamaño del vocabulario del modelo, cuantos tokens soporta.

Pero este es el tensor de salida, por qué tiene la forma de la secuencia de entrada?, porque cada posición en la salida corresponde a la la predicción del siguiente token de esa posición en la secuencia de entrada. En otras palabras, lo que obtenemos como predicción, es una secuencia de igual tamaño a la de entrada, movida un token hacia adelante, lo que efectivamente nos predice un solo token a la vez y por ende, el token que nos insteresa, es el último.

Lo que obtenemos en este tensor es además los logits de TODO el vocabulario del modelo, con los cuales podemos calcular las probabilidades de que cada uno sea el que continue en la secuencia. Hay varias formas de decodificar el siguiente token, la más fácil de implementar sería una decodificación codiciosa (greedy) del siguiente token, que consiste simplemente en seleccionar el token con la probabilidad más alta. Este es un enfoque simple y efectivo para algunos casos, pero al mismo tiempo sufre de poca variabilidad e incluso puede caer en generación repetitiva.

Otra opción es el muestreo, ya que justamente podemos obtener probabilidades del siguiente token, lo más lógico sería muestrear con esas opciones de probabilidad, de este modo podemos obtener mayor diversidad a la hora de generar el texto, al costo eso si de que haya mayor aleatoridad ya que se le daría la oportunidad a incluso tokens con baja probabilidad, de ser seleccionados.

Otra opción podría ser un balanceo entre una decodificación greedy y una por muestreo, en función de otro hiperparámetro que podemos definir. Esta sería una técnica muy común en el contexto de Reinforcement Learning llamade e-greedy. Se hace la aclaración de que en este ejemplo no harémos nada de RL, solamente se hace mención de esta técnica para balancear entre explotación y exploración.

In [32]:
import torch.nn as nn
import numpy as np
import pandas as pd
from typing import Optional, Tuple
from transformers.tokenization_utils_base import PreTrainedTokenizerBase


def generate(
        model: nn.Module,
        tokenizer: PreTrainedTokenizerBase,
        start: str,
        max_length: int = 100,
        eps: float = 0.5,
        top_n: int = 5,
        return_iterations: bool = False,
        device: str = "cpu") -> Tuple[str, Optional[pd.DataFrame]]:

    output = [start]
    iterations = []
    with torch.no_grad():
        input_ids = tokenizer(output[-1], return_tensors='pt')['input_ids'].to(device)
        for _ in range(max_length):
            # Tomamos los logits producidos por la última capa del modelo
            # Estos corresponden al siguiente token por cada posición de la cadena
            logits = model(input_ids=input_ids).logits
            # Por lo tanto, el que nos interesa es el último, que correspondería a la
            # predicción del siguiente token después del final de la cadena original
            # A este aplicamos un softmax para obtener las probabilidades por cada
            # token del vocabulario para estar presente en la cadena.
            probs = torch.softmax(logits[0, -1, :], dim=-1)
            # Simplemente ordenamos por probabilidad de forma descendente
            sorted_tokens = torch.argsort(probs, dim=-1, descending=True)

            # Utilizamos una politica tipo e-greedy para obtener el siguiente token de la secuencia
            # Un eps>=1 quiere decir que siempre se va seleccionar el token de forma 'greedy', es decir
            # siempre se toma el token con probabilidad más alta.

            # Un eps=0 quiere decir que siempre se va a muestrear el siguiente token en función
            # de las probabilidades de cada token

            # Un 0<eps<1 va a balancear de forma binomial entre tomar el token con la
            # probabilidad más alta y muestrear el token en función de sus probabilidades.
            if np.random.random_sample(1)[0] < eps:
                # Se toma el mejor token
                next_token = sorted_tokens[0].unsqueeze(dim=0)
            else:
                # Se muetrea el token de la probabilidad de distribución
                next_token = torch.multinomial(probs, 1)

            if return_iterations:
                # Mantenemos pista de todas las iteraciones para análisis
                iteration = {'input': ''.join(output)}
                best_n = sorted_tokens[:top_n].cpu().tolist()
                choices = {f'Choice #{choice+1}': f'{tokenizer.decode(token)} ({prob:.4f})' for choice, (token, prob) in enumerate(zip(best_n, probs[best_n].cpu().tolist()))}
                iteration.update(choices)
                iterations.append(iteration)

            output.append(tokenizer.decode(next_token))
            input_ids = torch.cat([input_ids, next_token.unsqueeze(dim=0)], dim=-1)

        output_text = ''.join(output)
        if not return_iterations:
            return output_text, None
        else:
            df = pd.DataFrame(iterations)
            return output_text, df

Ahora observemos que pasa cuando generamos texto con nuestra función y algunos parámetros.

Primero, observemos que pasa cuando pasamos un `eps=1` que quiere decir que la generación va a ser de tipo greedy:

In [33]:
output_text, iterations_df = generate(model, tokenizer, text, max_length=15, eps=1.0, top_n=10, return_iterations=True, device=device)
print(output_text)
iterations_df.head(15)

Había una vez más, el hombre que había sido su padre, el que había sido su


,input,Choice #1,Choice #2,Choice #3,Choice #4,Choice #5,Choice #6,Choice #7,Choice #8,Choice #9,Choice #10
0,Había una vez,más (0.4087),que (0.1055),en (0.0727),", (0.0490)",allí (0.0099),se (0.0087),. (0.0082),dentro (0.0081),a (0.0078),al (0.0078)
1,Había una vez más,", (0.5322)",en (0.0701),. (0.0519),la (0.0194),que (0.0191),: (0.0174),el (0.0162),se (0.0157),a (0.0142),y (0.0113)
2,"Había una vez más,",el (0.0491),se (0.0456),la (0.0409),en (0.0339),y (0.0301),no (0.0299),había (0.0205),me (0.0194),su (0.0158),los (0.0157)
3,"Había una vez más, el",hombre (0.0230),hecho (0.0225),señor (0.0173),joven (0.0165),recuerdo (0.0157),muchacho (0.0120),mundo (0.0119),corazón (0.0105),cuerpo (0.0095),deseo (0.0087)
4,"Había una vez más, el hombre",que (0.1364),se (0.0979),de (0.0590),había (0.0475),no (0.0358),le (0.0276),estaba (0.0216),la (0.0205),del (0.0204),al (0.0195)
5,"Había una vez más, el hombre que",había (0.1757),se (0.0684),estaba (0.0556),le (0.0475),la (0.0471),tenía (0.0348),lo (0.0310),me (0.0221),no (0.0146),amaba (0.0139)
6,"Había una vez más, el hombre que había",sido (0.0862),estado (0.0856),visto (0.0478),hablado (0.0323),conocido (0.0255),intentado (0.0232),hecho (0.0230),matado (0.0214),en (0.0184),entrado (0.0129)
7,"Había una vez más, el hombre que había sido",su (0.1692),", (0.0989)",el (0.0616),. (0.0441),antes (0.0383),en (0.0360),y (0.0337),capaz (0.0231),durante (0.0221),siempre (0.0183)
8,"Había una vez más, el hombre que había sido su",padre (0.1336),compañero (0.1296),amigo (0.1290),hermano (0.0368),marido (0.0357),primer (0.0353),hijo (0.0245),jefe (0.0181),mejor (0.0178),amante (0.0173)
9,"Había una vez más, el hombre que había sido su...",", (0.1805)",había (0.1334),. (0.1022),y (0.0761),se (0.0562),no (0.0505),en (0.0381),le (0.0301),durante (0.0280),era (0.0277)


Observamos como el input progresa a la vez que las opciones de tokens que hay. Sin importar cuantas veces invoquemos a la función con los mismos parámetros, siempre vamos a obtener los mismos resultados.

Ahora, observemos que pasa si introducimos exploración al reducir el `eps=0.5`, lo cual nos dice que aproximadamente la mitad de las veces va a elegir el siguiente token muestreando y la otra mitad explotando.

In [34]:
output_text, _ = generate(model, tokenizer, text, max_length=100, eps=0.5, device=device)
print(output_text)

Había una vez más, su fin había sido el de la maternidad. A veces permanecía en la cama, como si no hubiera pasado nada. 

Sarah estaba tumbada en la cama, encogida, adormilada. 

—Debe de haber dormido mal —dijo él. 

—No, no lo creo. 

—No soy tonta. Sé que no podría…

Él le acarició el pelo. 

—Has tenido un día muy duro. No había dormido mal… me han dicho


En este caso, cada vez que invoquemos a la función, vamos a obtener una respuesta diferente, a veces más coherente y otras veces menos. Vale la pena realizar varias pruebas para observar los resultados hasta encontrar un balance.

### Generando texto con las utilidades del modelo

Ahora, la clase de Huggingface implementa la función `generate` que hace la labor de generación por nosotros, incluyendo las opciones de muestreo y explotación como hemos observado. Solo que además permite otra serie de parámetros y opciones para controlar la generación de texto. Para más información se recomienda estudiar:

- [Natural Language Processing with Transformers: Building Language Applications With Hugging Face](https://www.amazon.com/Natural-Language-Processing-Transformers-Applications/dp/1098103246), Capitulo 5
- https://huggingface.co/docs/transformers/v4.41.3/en/main_classes/text_generation#transformers.GenerationConfig
- https://huggingface.co/docs/transformers/v4.41.3/en/main_classes/text_generation#transformers.GenerationMixin.generate

In [35]:
output = model.generate(tokens, pad_token_id=tokenizer.eos_token_id, max_length=100, do_sample=True, temperature=0.5, top_k=0)
print(tokenizer.decode(output[0]))

Había una vez más, se había dado cuenta de que tenía el aspecto de un hombre de negocios. 

—No te preocupes, no te preocupes. —Se sentó en su sillón y cogió un libro de la estantería—. Enséñame un libro. 

—Te lo enseñaré. 

—No, no me lo enseñes. 

—No te lo enseñaré. 

—Entonces, ¿por qué no te lo enseño? 

—No me lo ense


## Fine tuning

Ahora, intentemos hacer fine tuning a nuestro modelo. Intentemos entrenarlo en un corpus de chistes en idioma español y ver como la narrativa de su output cambia.

##### Nota
Lastimosamente, este dataset es muy pequeño y la distribución del texto es muy diferente a la distribución de texto con la cual fue entrenado el modelo original, por lo que no se esperan resultados significativamente ejemplares. Sin embargo, el objetivo es observar como cambia la generación del texto una vez lo entrenamos en un conjunto especializado.

## Desde la librería datasets de Hugging Face se importa la función load_dataset, la cual permite descargar y preparar de forma rápida y sencilla distintos conjuntos de datos.

In [36]:
# Importa la función load_dataset desde la librería 'datasets' de HuggingFace.
# Esta función permite descargar y preparar rápidamente conjuntos de datos para NLP y ML.
from datasets import load_dataset

# Usa load_dataset para descargar y cargar el dataset "josuealdana/latin_lyrics" desde HuggingFace Hub.
# El resultado es un objeto tipo DatasetDict que contiene los splits y metadatos del dataset cargado.
dataset = load_dataset("josuealdana/latin_lyrics")

# Muestra la estructura general y los detalles del dataset.
dataset

DatasetDict({
    train: Dataset({
        features: ['conversations'],
        num_rows: 3759
    })
})

In [37]:
dataset['train'][0]

{'conversations': "[{'role': 'system', 'content': 'Escribe una canción en el estilo de Carla Morrison, en el género de Mexican Indie.'}, {'role': 'user', 'content': 'Título de la canción: Disfruto Francistyle Version'}, {'role': 'assistant', 'content': 'Quiero disfrutar contigo\\nEl tiempo y el espacio\\nAcariciarte el pelo\\nHasta que se te ponga la piel de gallina\\nOye vida mía si voy a dormir a tu lado\\nQuisiera que al abrir los ojos estés a mi lado\\nMe complace amarte\\nDisfruto acariciarte\\nY ponerte a dormir\\nEs escalofriante\\nTenerte de frente…\\n…hacerte sonreír\\nDaría cualquier cosa…\\nPor tan primorosa\\nPor estar siempre aquí\\nY entre todas esas cosas\\nDéjame querеrte…\\nEntrégate a mí\\nNo te fallaré\\nContigo yo quiеro envejecer\\nQuiero darte un beso\\nPerder contigo mí tiempo\\nGuardar tus secretos\\nCuidar tus momentos\\nAbrazarte\\nEsperarte, adorarte\\nTenerte paciencia\\nTu locura es mi ciencia\\nAhhh… ohhh…\\n…\\nNo hay duda mi amor\\nMi vida cambió\\nDe gr

Este código carga el dataset latin_lyrics de Hugging Face, lo transforma en un DataFrame y extrae de cada registro el título y la letra de la canción (ya sea desde columnas existentes o desde la estructura conversations). Luego construye un dataset limpio con columnas estandarizadas y lo muestra/guarda para su uso posterior.

In [38]:
from datasets import load_dataset
import pandas as pd
import ast

# -------- Utilidades --------
def extract_title_and_song_from_conversations(conv_str):
    """Para filas que traen 'conversations' como string con [{'role':..., 'content':...}, ...]."""
    try:
        conv_list = ast.literal_eval(conv_str)
    except Exception:
        return "", ""
    title, song = "", ""
    for item in conv_list:
        if isinstance(item, dict):
            role = item.get("role", "")
            content = item.get("content", "")
            if role == "user" and "Título de la canción" in content:
                parts = content.split("Título de la canción:")
                if len(parts) > 1:
                    title = parts[1].strip()
            elif role == "assistant":
                song = content.strip()
    return title, song

def build_songs_dataset_from_df(df):
    """
    Crea un nuevo DataFrame con columnas: ['title','song','full_song'].
    Detecta automáticamente columnas típicas de letras y título.
    Si existe 'conversations', extrae de ahí.
    """
    df = df.copy()

    # Caso A: dataset estilo 'latin_lyrics' (tiene columnas de título y letra)
    title_col = next((c for c in ['title','song','track','name'] if c in df.columns), None)
    lyrics_col = next((c for c in ['lyrics','lyric','text','content'] if c in df.columns), None)

    if 'conversations' in df.columns:
        titles, songs = [], []
        for conv in df['conversations']:
            t, s = extract_title_and_song_from_conversations(conv)
            titles.append(t)
            songs.append(s)
        out = pd.DataFrame({'title': titles, 'song': songs})

    elif title_col and lyrics_col:
        # Usar directamente las columnas del dataset
        out = pd.DataFrame({
            'title': df[title_col].astype(str).fillna(''),
            'song':  df[lyrics_col].astype(str).fillna(''),
        })

    elif lyrics_col:
        # Si solo hay letra, intenta construir título vacío
        out = pd.DataFrame({
            'title': '',
            'song': df[lyrics_col].astype(str).fillna(''),
        })

    else:
        raise ValueError(
            "No encuentro columnas de letra/título. Revisa df.columns. "
            "Intenta mapear manualmente: title_col='...', lyrics_col='...'."
        )

    out['full_song'] = out.apply(
        lambda r: (f"{r['title']}\n{r['song']}").strip() if r['title'] or r['song'] else "",
        axis=1
    )

    # Índice desde 1 con nombre 'indice'
    out.index = range(1, len(out) + 1)
    out.index.name = 'indice'
    return out

# -------- Carga de tu dataset --------
# Esto devuelve un DatasetDict con splits. Ej.: {'train': Dataset(...)}
ds = load_dataset("josuealdana/latin_lyrics")

# Elige un split. Si no sabes cuál, toma el primero disponible.
split_name = next(iter(ds.keys()))           # por ejemplo 'train'
hf_dataset = ds[split_name]                   # Dataset de Hugging Face del split
df = hf_dataset.to_pandas()                   # -> DataFrame

# (Opcional) mira qué columnas trae para confirmar:
# print(df.columns)

# Construir el nuevo dataset de canciones (título + letra)
songs_df = build_songs_dataset_from_df(df)

# Ver resultado
print(songs_df.head(10))

# Guardar si quieres:
# songs_df.to_csv("canciones_extraidas.csv", index=True, encoding="utf-8")

                                      title  \
indice                                        
1              Disfruto Francistyle Version   
2                                   Eres Tu   
3                        Gemitaiz Ciao Baby   
4                                  Disfruto   
5           Karol G And  Manana Sera Bonito   
6       Macklemore And Ryan Lewis The Train   
7                             Hasta La Piel   
8                     Billie Eilish Eres Tu   
9                                 Te Regalo   
10                                Todo Pasa   

                                                     song  \
indice                                                      
1       Quiero disfrutar contigo\nEl tiempo y el espac...   
2       [Letra de "Eres Tú"]\n[Verso 1]\nHoy desperté ...   
3       [Testo di "Ciao Baby"]\n[Intro]\nDisfruto mira...   
4       [Letra de "Disfruto"]\n[Verso 1]\nMe complace ...   
5       [Letra de "MAÑANA SERA BONITO"]\n[Refrán: Carl...   
6       

In [39]:
from datasets import Dataset, DatasetDict

# Convertir el DataFrame en un Dataset de Hugging Face
songs_hf = Dataset.from_pandas(songs_df)

# Si quieres crear un DatasetDict (por ejemplo con un split 'train')
songs_datasetdict = DatasetDict({
    "train": songs_hf
})

# Verificar
print(songs_datasetdict)
print(songs_datasetdict["train"][2])  # primer registro

DatasetDict({
    train: Dataset({
        features: ['title', 'song', 'full_song'],
        num_rows: 3759
    })
})
{'title': 'Gemitaiz Ciao Baby', 'song': '[Testo di "Ciao Baby"]\n[Intro]\nDisfruto mirarte\nCada movimiento\nUn vicio que tengo\nUn gusto, valorarte\nNunca olvidarte\nEntregarte mis tiempos\n[Strofa 1]\nTi sfrutto, lo sai che non posso restare muto\nChe mi manchi e nella notte come un lupo\nNei campi e stenderci a guardare gli UFO\nTi amo, passo e chiudo\nAnche quando poi mi fai impazzire e litighiamo\nE tutte le cose belle le eliminiamo\nDopo ridi e aspetti che ti richiamo\nE ti proietti bella come la luna sopra quei tetti, yeah\nMetto da parte tutto se so che vieni\nTi sento senza vederti come gli alieni\nIl tuo nome scritto in argento sopra quei treni\n[Ritornello]\nDisfruto mirarte\nCada movimiento\nUn vicio que tengo\nUn gusto, valorarte\nNunca olvidarte\nEntregarte mis tiempos\nNo te fallaré\nContigo yo quiero envejecer\n[Strofa 2]\nMusica per l\'anima\nQuando sei

In [40]:
songs_datasetdict.set_format('pandas')
df = songs_datasetdict['train'].to_pandas()
df.head(10)

,title,song,full_song
0,Disfruto Francistyle Version,Quiero disfrutar contigo\nEl tiempo y el espac...,Disfruto Francistyle Version\nQuiero disfrutar...
1,Eres Tu,"[Letra de ""Eres Tú""]\n[Verso 1]\nHoy desperté ...","Eres Tu\n[Letra de ""Eres Tú""]\n[Verso 1]\nHoy ..."
2,Gemitaiz Ciao Baby,"[Testo di ""Ciao Baby""]\n[Intro]\nDisfruto mira...","Gemitaiz Ciao Baby\n[Testo di ""Ciao Baby""]\n[I..."
3,Disfruto,"[Letra de ""Disfruto""]\n[Verso 1]\nMe complace ...","Disfruto\n[Letra de ""Disfruto""]\n[Verso 1]\nMe..."
4,Karol G And Manana Sera Bonito,"[Letra de ""MAÑANA SERA BONITO""]\n[Refrán: Carl...","Karol G And Manana Sera Bonito\n[Letra de ""MA..."
5,Macklemore And Ryan Lewis The Train,[Verse 1: Macklemore]\nPretty face tryna chase...,Macklemore And Ryan Lewis The Train\n[Verse 1:...
6,Hasta La Piel,"[Letra de ""Hasta La Piel""]\n[Verso 1]\nNo te q...","Hasta La Piel\n[Letra de ""Hasta La Piel""]\n[Ve..."
7,Billie Eilish Eres Tu,"[Letra de ""Eres Tú""]\n[Verse 1]\nHoy desperté ...","Billie Eilish Eres Tu\n[Letra de ""Eres Tú""]\n[..."
8,Te Regalo,[Verse 1]\nDéjame tomarte de la mano\nDéjame m...,Te Regalo\n[Verse 1]\nDéjame tomarte de la man...
9,Todo Pasa,Letra de Todo Pasa\nMi vista es distinta\nEs m...,Todo Pasa\nLetra de Todo Pasa\nMi vista es dis...


In [41]:
df['Palabras por cancion'] = df['full_song'].str.split().apply(len)
median = df['Palabras por cancion'].median()
print(f"La mediana de palabras por canción es de {median}")

La mediana de palabras por canción es de 195.0


Aquí podemos observar que la mediana de longitud en terminos de palabras es de 199. Esto es esperado, pues las canciones son cortas por naturaleza. Por otra parte, es bastante claro que el corpus original del modelo pre-entrenado contenía texto muy diferente a este, por lo que la calidad de los resultados, sin hacer mayores modificaciones puede que no sea buena.

Sin embargo, a manera demostrativa, continuarémos con el ejercicio, prepararémos el conjunto de datos para entrenamiento.

In [42]:
max_len = int(median * 1.10)
print(f"El nuevo valor de max_len es: {max_len}")

El nuevo valor de max_len es: 214


Para dar un poco de mayor longitud al texto generado dado que, el tamaño las canciones del dataser pueden se inferiories o mayores a la media se decide darle un 10% de longitud adicional del valor de la mediana.

In [43]:
def preprocess_function(max_len):
    def _preprocess_function(examples):
        return tokenizer(examples['full_song'], max_length=max_len, truncation=True, padding='max_length')
    return _preprocess_function


Los modelos GPT no esperan otra cosa más que los `input_ids`, por lo que retirarémos todas las demás columnas del dataset ya que no nos son de utilidad en este momento.

In [44]:
from datasets import Dataset, DatasetDict

# Convert the DataFrame back to a Hugging Face Dataset, keeping the 'full_song' column name
dataset_hf = Dataset.from_pandas(df[['full_song']])

# If you want to maintain the DatasetDict structure
dataset = DatasetDict({
    'train': dataset_hf
})

tokenized_dataset = dataset['train'].map(lambda examples: preprocess_function(max_len=max_len)(examples), batched=True)
tokenized_dataset = tokenized_dataset.remove_columns([col for col in tokenized_dataset.column_names if col != 'input_ids'])
tokenized_dataset = tokenized_dataset.train_test_split(train_size=0.9)
tokenized_dataset.set_format('torch')
tokenized_dataset

Map:   0%|          | 0/3759 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids'],
        num_rows: 3383
    })
    test: Dataset({
        features: ['input_ids'],
        num_rows: 376
    })
})

Finalmente procedemos a definir el entrenamiento. Observaremos que es muy similar a como entrenamos a BERT.

In [45]:
from transformers import DataCollatorForLanguageModeling
from transformers import Trainer, TrainingArguments


batch_size = 8 if IN_COLAB else 2
logging_steps = len(tokenized_dataset['train']) // batch_size
# Definimos los parámetros globales de entrenamiento
training_args = TrainingArguments(
    output_dir='./hf-gpt',
    overwrite_output_dir=True,
    num_train_epochs=5,
    learning_rate=2e-5,
    per_device_eval_batch_size=batch_size,
    per_device_train_batch_size=batch_size,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    disable_tqdm=False,
    logging_steps=logging_steps,
    report_to='none'
)

# Y definimos el entrenador, especificando el modelo, datasets y el tokenizador
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['test'],
    tokenizer=tokenizer
)

In [46]:
%%time
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.


Epoch,Training Loss,Validation Loss
1,3.229000,2.998927
2,2.917100,2.903740
3,2.774100,2.862247
4,2.691400,2.839373
5,2.640800,2.834564


There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


CPU times: user 16min 54s, sys: 9.61 s, total: 17min 3s
Wall time: 19min 32s


TrainOutput(global_step=2115, training_loss=2.849745194004501, metrics={'train_runtime': 1171.4041, 'train_samples_per_second': 14.44, 'train_steps_per_second': 1.806, 'total_flos': 1847319356160000.0, 'train_loss': 2.849745194004501, 'epoch': 5.0})

El modelo logró entrenar en 5 épocas reduciendo la pérdida de entrenamiento de 3.229 → 2.64 y la de validación de 2.998 → 2.83, lo que indica mejora real en el aprendizaje con buen poder de generalización; pese a la advertencia por una clave faltante, el entrenamiento finalizó de forma estable con una pérdida final de ~2.834.

Ahora observemos los resultados.

In [47]:
text = "Bajo la luna de Cali,"
tokens = tokenizer(text, return_tensors='pt')['input_ids'].to(device) # <<< mover tokens al mismo device

output = model.generate(tokens, pad_token_id=tokenizer.eos_token_id, max_length=max_len, do_sample=True, temperature=0.8)
print(tokenizer.decode(output[0]))

Bajo la luna de Cali,
Letra de "Ricky"
[Verso 1]
Yo, que por casualidad, me fui,
Vengo a decirte que en mis ojos, hay tristeza
Que el tiempo es muy cruel,
Que todo lo que me une
Es un dolor inmenso, que me mata
Y que el alma se va, que todo lo que me une
Es un dolor inmenso, que me mata
Y que el alma, todo lo que me une
Es un dolor inmenso, que me mata
Y que el alma, todo lo que me une
Es un dolor inmenso, que me mata
Y que el alma, todo lo que me une
Es un dolor inmenso, que me mata
Y que el alma, todo lo que me une
Es un dolor inmenso, que me mate
Y que el alma, todo lo que me une
Es un dolor inmenso, que me mata
Y que el alma, todo lo que me une
Es un dolor inmenso, que me mata
Y que el alma


In [48]:
output_text, _ = generate(model, tokenizer, text, max_length=max_len, eps=0.2, device=device)
print(output_text)

Bajo la luna de Cali, los chefs de la ciudad de Bogotá, y el asombro de saber que el premio es una oportunidad para el Nani? Hasta que encontremos el lugar exacto donde se llevó las palmas de mis manos (y te las encontraste entre mis brazos)
Y volvamos al asunto del que te hablé
Y si existe un alma en esta ciudad, que sea tu
Y si hay un alma en esta ciudad, la que sea tu
Y si existe un alma en esta ciudad, que sea tu
Y si existe una alma en esta ciudad, la que sea tu
Y si existe una alma en esta ciudad, la que sea tu
Y si hay un alma en esta ciudad, la que sea tu
Y si existe un alma en esta ciudad, que sea tu
Y si hay un alma en esta ciudad, la que sea tu
Y si hay un alma en esta ciudad, la que sea tu
Y si hay un alma en este infierno, que sea tu
Y si hay un alma en esta ciudad, la que sea tu
Y si hay un alma


In [49]:
!pip install gtts pydub

Este código convierte la letra de una canción en voz mediante gTTS, la sincroniza con una pista instrumental de salsa usando pydub, ajusta la duración y volumen de la base musical, y finalmente mezcla ambos audios para generar un archivo MP3 con la canción resultante.

In [50]:
# Instala requerimientos:
# pip install gtts pydub

from gtts import gTTS
from pydub import AudioSegment

# Texto de tu canción (letra de la canción como string)
letra = output_text

# Paso 1: convierte la letra en voz (gTTS usa por defecto voz hablada, acentúa un poco el ritmo con breaks)
tts = gTTS(text=letra, lang='es')
tts.save("voz_letra.mp3")

# Paso 2: carga la pista instrumental de salsa (debes tener tu propio archivo por derechos de autor)
# Por ejemplo, 'salsa_base.mp3' debe ser un loop/instrumental de salsa-
!wget  https://cdn.pixabay.com/download/audio/2025/06/13/audio_48f211ff6e.mp3?filename=salsa-latin-party-music-349496.mp3 -O salsa_base.mp3
pista_salsa = AudioSegment.from_file("/content/salsa_base.mp3")
voz = AudioSegment.from_file("voz_letra.mp3")

# Ajusta duración de la pista para igualar la voz
if len(pista_salsa) < len(voz):
    repeticiones = int(len(voz) / len(pista_salsa)) + 1
    pista_salsa = pista_salsa * repeticiones
pista_salsa = pista_salsa[:len(voz)]

# Opcional: baja el volumen de la base musical para que destaque la voz
pista_salsa = pista_salsa - 8

# Mezcla voz con base
mezcla = pista_salsa.overlay(voz)

# Exporta el resultado
mezcla.export("cancion_salsa_final.mp3", format="mp3")


--2025-09-22 01:36:53--  https://cdn.pixabay.com/download/audio/2025/06/13/audio_48f211ff6e.mp3?filename=salsa-latin-party-music-349496.mp3
Resolving cdn.pixabay.com (cdn.pixabay.com)... 172.64.147.160, 104.18.40.96, 2606:4700:4400::ac40:93a0, ...
Connecting to cdn.pixabay.com (cdn.pixabay.com)|172.64.147.160|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3607296 (3.4M) [audio/mpeg]
Saving to: ‘salsa_base.mp3’

salsa_base.mp3      100%[===================>]   3.44M  --.-KB/s    in 0.1s    

2025-09-22 01:36:54 (31.7 MB/s) - ‘salsa_base.mp3’ saved [3607296/3607296]



<_io.BufferedRandom name='cancion_salsa_final.mp3'>

In [51]:
!apt-get update
!apt-get install -y espeak
!pip install pyttsx3 pydub

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Fetched 257 kB in 4s (63.8 kB/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provi

Este código convierte la letra de una canción en voz en español utilizando gTTS y la combina con una pista instrumental de salsa mediante pydub. Para ello, ajusta la duración de la base musical (repitiéndola si es más corta que la voz), reduce su volumen para resaltar la voz y finalmente exporta el resultado como cancion_salsera_final.mp3, generando así una versión cantada con acompañamiento instrumental.

In [52]:
# Instalar librerias necesarias:
# pip install gtts pydub

from gtts import gTTS
from pydub import AudioSegment
import numpy as np # Although numpy is imported, it's not strictly needed for this gtts approach

# Texto de tu canción (letra de la canción como string)
letra = output_text

# Paso 1: convierte la letra en voz usando gTTS
tts = gTTS(text=letra, lang='es')
tts.save("voz_letra.mp3")

# Paso 2: carga la pista instrumental de salsa (debes tener tu propio archivo por derechos de autor)
# For example, 'salsa_base.mp3' should be a salsa loop/instrumental.
# Make sure you have downloaded the file to the correct path or update the path below.
# If you don't have a file, you can use the wget command from the previous successful cell.
!wget https://cdn.pixabay.com/download/audio/2025/06/13/audio_48f211ff6e.mp3?filename=salsa-latin-party-music-349496.mp3 -O salsa_base.mp3
pista_salsa = AudioSegment.from_file("/content/salsa_base.mp3")

# Cargar audio de voz generado
voz_salsera = AudioSegment.from_file("voz_letra.mp3")

# Ajustar duración pista salsa para que coincida con la voz
if len(pista_salsa) < len(voz_salsera):
    repeticiones = int(len(voz_salsera) / len(pista_salsa)) + 1
    pista_salsa = pista_salsa * repeticiones
pista_salsa = pista_salsa[:len(voz_salsera)]

# Bajar volumen instrumental para que destaque la voz
pista_salsa = pista_salsa - 10

# Mezclar la voz con la pista instrumental
mezcla = pista_salsa.overlay(voz_salsera)

# Guardar mezcla final
mezcla.export("cancion_salsera_final.mp3", format="mp3")

print("Audio generado como 'cancion_salsera_final.mp3'")

--2025-09-22 01:37:24--  https://cdn.pixabay.com/download/audio/2025/06/13/audio_48f211ff6e.mp3?filename=salsa-latin-party-music-349496.mp3
Resolving cdn.pixabay.com (cdn.pixabay.com)... 104.18.40.96, 172.64.147.160, 2a06:98c1:3107::6812:2860, ...
Connecting to cdn.pixabay.com (cdn.pixabay.com)|104.18.40.96|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3607296 (3.4M) [audio/mpeg]
Saving to: ‘salsa_base.mp3’

salsa_base.mp3      100%[===================>]   3.44M  20.1MB/s    in 0.2s    

2025-09-22 01:37:24 (20.1 MB/s) - ‘salsa_base.mp3’ saved [3607296/3607296]

Audio generado como 'cancion_salsera_final.mp3'


En conclusión, la letra presenta imágenes claras y un ritmo repetitivo que pueden aportar cercanía e interés musical; sin embargo, el exceso de repeticiones, la falta de claridad en la narrativa y el cierre inconcluso hacen que pierda fuerza poética y deje una sensación de inacabado. Con una edición que equilibre la repetición con mayor desarrollo narrativo y un final definido, podría transformarse en una canción más sólida y expresiva.


## Conclusiones:
- A pesar de que estamos generando texto con GPT-2, el modelo busca ofrecer una aproximación coherente y significativa.
- Aunque el modelo logra imitar patrones de escritura del dataset, aún presenta limitaciones en estructura poética y en la coherencia narrativa a lo largo de toda la letra.
- La calidad y el estilo de las letras generadas dependen directamente del dataset de entrenamiento. Un dataset más robusto (con géneros específicos o canciones con estructura clara) puede llevar a resultados más consistentes y profesionales.

- A diferencia de otras tareas de NLP, la calidad de las letras generadas requiere también una evaluación artística y subjetiva, no solo métricas cuantitativas, ya que lo “bueno” en una canción depende del gusto musical y la intención creativa.

- En los modelos generativos, tiende a ver un dilema de tipo exploración-explitación, al explotar los resultados, podemos ser más precisos, per al mismo tiempo más monótonos, mientras que explorando podemos ser más creativos y diversos, pero al mismo tiempo terminar con texto incoherente, difuso o alucinante. Es necesario evaluar la tarea a la mano para escoger el ajuste adecuado entre estas dos técnicas de decodificación.
- Los modelos generativos de texto no son más que una gran probabilidad de distribución y esta a su vez es completamente dependiente de los datos con los que fue entrenada. Es aquí donde se hace sumamente importante obtener y curar los conjuntos de datos con los que se entrena, de lo contrario se puede terminar con un modelo de mala calidad para la tarea en especifico.
- Diferentes estrategias de decodificación entregan resultados diferentes, vale la pena hacer una exploración de los resultados y ajustar los hiperparámetros para obtener los resultados deseados según el objetivo.